# Archive — Old Recommend Functions

This notebook preserves earlier versions of the recommendation function for reference.
These are **not used in the main project** (`anime_project.ipynb`) anymore.
They were replaced by the Phase 1 `recommend()` function which adds fuzzy search,
genre filtering, min-votes filtering, and year-range filtering.

---

## Why Keep These?

In software engineering and data science, it's useful to keep old versions around so you can:
- **Compare outputs** — did the new version actually improve things?
- **Trace your thinking** — each version shows a decision you made and why
- **Roll back** — if the new approach has a bug, you have a working baseline

In professional teams this is handled by **git version control** (every old version lives in git history).
For a learning project, a separate archive notebook serves the same purpose.

---

## Version 1 — Original `recommend()`

**What it did:** Exact title match only. Used the original feature matrix (`feature_df`) with
`Rating Norm * 3` and `Votes Norm * 2` weights.

**Problem:** Two issues —
1. Exact match meant any typo or missing punctuation returned nothing
2. The inflated quality weights made popular anime cluster together regardless of genre
   (e.g. Sailor Moon returned Action/Adventure shows instead of Romance/Fantasy)

**Superseded by:** Phase 1 `recommend()` in `anime_project.ipynb`

In [ ]:
def recommend_v1(title, n=10):
    # Find the row number for this anime
    matches = feature_df[feature_df['Title'].str.lower() == title.lower()]

    if matches.empty:
        print(f"'{title}' not found. Check the spelling.")
        return

    idx = matches.index[0]

    # Get similarity scores for this anime against all others
    scores = list(enumerate(similarity_matrix[idx]))

    # Sort by similarity, highest first, skip the first one (itself)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]

    # Build result table
    results = []
    for i, score in scores:
        results.append({
            'Title': feature_df.iloc[i]['Title'],
            'Similarity': round(score, 3),
            'Rating': round(df_clean.iloc[i]['User Rating'], 1),
            'Genres': df_clean.iloc[i]['Genre']
        })

    return pd.DataFrame(results)

# recommend_v1("Attack on Titan")

---

## Version 2 — `recommend2()` (Genre Weight Fix)

**What it did:** Same exact-match lookup, but used a rebuilt feature matrix (`feature_df2`)
where genre columns were weighted at `3x` and quality features brought back to `1x`.
This made genre the primary similarity driver instead of popularity tier.

**Problem:** Still used exact title matching — no tolerance for typos.
Also had no filtering options (genre, year, votes).

**What improved over V1:** Genre recommendations became much more accurate.
Spirited Away correctly returned Family/Adventure anime instead of whatever
was in the same popularity tier.

**Superseded by:** Phase 1 `recommend()` in `anime_project.ipynb`

In [ ]:
def recommend_v2(title, n=10):
    matches = feature_df2[feature_df2['Title'].str.lower() == title.lower()]
    if matches.empty:
        print(f"'{title}' not found. Check the spelling.")
        return
    idx = matches.index[0]
    scores = sorted(enumerate(similarity_matrix2[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    results = [{
        'Title': feature_df2.iloc[i]['Title'],
        'Similarity': round(score, 3),
        'Rating': round(df_clean.iloc[i]['User Rating'], 1),
        'Genres': df_clean.iloc[i]['Genre']
    } for i, score in scores]
    return pd.DataFrame(results)

# recommend_v2("Attack on Titan")
# recommend_v2("Spirited Away")
# recommend_v2("Sailor Moon")

---

## Evolution Summary

| Version | Fuzzy Search | Genre Filter | Min Votes | Year Filter | Genre Weighting |
|---------|-------------|--------------|-----------|-------------|----------------|
| V1      | No          | No           | No        | No          | Rating 3x, Votes 2x (broken) |
| V2      | No          | No           | No        | No          | Genre 3x, Rating 1x (fixed) |
| **Phase 1** | **Yes** | **Yes**  | **Yes**   | **Yes**     | Genre 3x, Rating 1x |